# 05 · Alerts and Closing the Loop

**Module 07 · Production & Observability**

A dashboard is only useful if someone is looking at it. Nobody looks at it at 3 a.m. An **alert** turns "numbers you could look at" into "a message when a number is wrong".

And finding a problem is only half the job. The other half is making sure it **never happens again**. This notebook does both.

In [1]:
import copy
import json

import metrics
import observe
import shop
import guards
import desk
import evals

LOGS = "sample_logs"
report = metrics.load_report(LOGS)
print("Ready")

Ready


## 1. Alerts: limits on the numbers
An alert is a simple rule: *if this number crosses this limit, tell a person.* Pick limits from your own normal traffic. These are the ones we use:

*From `metrics.py`:*

```python
# The limits that, when crossed, mean "a person should look at this". Tune them to your own traffic.
DEFAULT_LIMITS = {
    "max_average_cost": 0.02,       # dollars per request
    "max_daily_cost": 5.00,         # dollars per day
    "max_p95_latency_ms": 25000,    # 95 out of 100 customers should wait less than this
    "max_error_rate": 0.05,         # 5% of requests failing is too many
    "max_block_rate": 0.15,         # a jump in blocked emails can mean an attack
    "max_human_rate": 0.30,         # too many handed to people means the desk is not coping
    "min_average_quality": 4.0,     # the judge's average score
    "max_thumbs_down_rate": 0.25,   # share of customer feedback that is negative
}


def check_alerts(report, limits):
    """Compare the report with the limits. Returns a list of warnings (an empty list means all is well)."""
    alerts = []
    if report["requests"] == 0:
        return ["No requests were logged."]

    total = report["requests"]

    if report["cost"]["average"] > limits["max_average_cost"]:
        alerts.append("Average cost per request is $" + format(report["cost"]["average"], ".4f")
                      + " (limit $" + format(limits["max_average_cost"], ".4f") + ")")

    for day in report["cost"]["by_day"]:
        if report["cost"]["by_day"][day] > limits["max_daily_cost"]:
            alerts.append("Spend on " + day + " was $" + format(report["cost"]["by_day"][day], ".2f")
                          + " (limit $" + format(limits["max_daily_cost"], ".2f") + ")")

    if report["latency_ms"]["p95"] > limits["max_p95_latency_ms"]:
        alerts.append("Slow: p95 latency is " + str(report["latency_ms"]["p95"]) + " ms (limit "
                      + str(limits["max_p95_latency_ms"]) + " ms)")

    if report["errors"] / total > limits["max_error_rate"]:
        alerts.append("Errors: " + str(report["errors"]) + " of " + str(total) + " requests failed")

    if report["guardrails"]["blocked"] / total > limits["max_block_rate"]:
        alerts.append("Blocked emails: " + str(report["guardrails"]["blocked"]) + " of " + str(total)
                      + ". Is someone attacking the desk?")

    if report["guardrails"]["sent_to_human"] / total > limits["max_human_rate"]:
        alerts.append("Handed to humans: " + str(report["guardrails"]["sent_to_human"]) + " of " + str(total))

    quality = report["quality"]
    if quality["graded"] > 0 and quality["average_score"] < limits["min_average_quality"]:
        alerts.append("Quality: the judge's average score is " + str(quality["average_score"])
                      + " (minimum " + str(limits["min_average_quality"]) + ")")

    votes = report["feedback"]["up"] + report["feedback"]["down"]
    if votes > 0 and report["feedback"]["down"] / votes > limits["max_thumbs_down_rate"]:
        alerts.append("Customers: " + str(report["feedback"]["down"]) + " of " + str(votes) + " ratings are thumbs down")

    return alerts
```

In [2]:
alerts = metrics.check_alerts(report, metrics.DEFAULT_LIMITS)

print("Alerts for the sample logs:")
for alert in alerts:
    print("!", alert)

Alerts for the sample logs:
! Blocked emails: 6 of 30. Is someone attacking the desk?


One alert fired: 20% of the emails were blocked (we sent attacks on purpose), and our limit is 15%. A person would look at the blocked requests and see whether it is a real attack or a limit that is set too low.

## 2. What would a bad day look like?
Our sample logs are healthy. To see the other alerts work, we **pretend**: we copy the report and make up bad numbers. (These numbers are invented, only to show the alerts. Never do this with real reports.)

In [3]:
pretend = copy.deepcopy(report)
pretend["cost"]["average"] = 0.045                 # a request now costs 3x more
pretend["cost"]["by_day"] = {"2026-11-22": 9.40}   # a day's spend explodes
pretend["latency_ms"]["p95"] = 41000               # the slow tail is very slow
pretend["errors"] = 6                              # 6 of 30 requests failed
pretend["quality"]["average_score"] = 3.4          # replies got worse
pretend["feedback"] = {"up": 4, "down": 9}         # customers are unhappy

for alert in metrics.check_alerts(pretend, metrics.DEFAULT_LIMITS):
    print("!", alert)

! Average cost per request is $0.0450 (limit $0.0200)
! Spend on 2026-11-22 was $9.40 (limit $5.00)
! Slow: p95 latency is 41000 ms (limit 25000 ms)
! Errors: 6 of 30 requests failed
! Blocked emails: 6 of 30. Is someone attacking the desk?
! Quality: the judge's average score is 3.4 (minimum 4.0)
! Customers: 9 of 13 ratings are thumbs down


Every alert says **what** crossed **which** limit, with numbers. An alert that says only "something is wrong" is useless at 3 a.m.

### Alerts need care
| Problem | What happens | The fix |
|---|---|---|
| **Too many alerts** | People start ignoring them ("alert fatigue") | Alert only on what needs a human, and tune the limits |
| **Too few** | You hear about problems from customers | Alert on cost, errors, latency, quality, and behaviour |
| **No owner** | Everyone assumes someone else will look | Each alert goes to a named person or channel |
| **Alerts without a next step** | Panic | Write a short checklist for each (see below) |

In a real system alerts are sent by your monitoring tool to Slack, email or a pager. Here the function returns a list, and **the service exposes it** at the admin endpoint `/metrics`.

## 3. When an alert fires: a checklist
1. **Flip the kill switch if customers are being hurt.** `POST /kill-switch` sends every email to a person, instantly. Investigate afterwards.
2. **Look at the dashboard.** Which number moved, and when?
3. **Find the worst requests.** The report lists them (thumbs down, low judge score, errors).
4. **Read their traces.** Which step went wrong?
5. **Check the versions.** Did `prompt_version` or `code_version` change when the numbers did?
6. **Fix it.** Then **add a test** so it can never come back (next section).

## 4. Closing the loop: a real problem becomes a test
This is the most valuable habit in the whole module. Every real failure is a free test case, because you now have:

```
a real request  →  a trace  →  a person decides what SHOULD have happened  →  an eval case  →  the suite
```

Take one of the requests the customers marked with a thumbs down. The trace stored the (redacted) email. A **person** decides what the right outcome was, here: *the desk should politely refuse, and mention the 30 days.*

In [4]:
traces = metrics.load_json_lines("traces.jsonl", LOGS)

flagged = None
for item in metrics.load_json_lines("feedback.jsonl", LOGS):
    if item["rating"] == "down":
        for record in traces:
            if record["request_id"] == item["request_id"]:
                if "return an item" in record["attributes"]["email_redacted"]:
                    flagged = record   # the customer who asked about returning an item after 45 days

print("Request:", flagged["request_id"])
print("Email  :", flagged["attributes"]["email_redacted"])
print("Reply  :", flagged["attributes"]["reply_redacted"])

Request: 9d8ebf424424
Email  : Can I return an item I bought 45 days ago?
Reply  : You cannot return an item bought 45 days ago, as the refund window is 30 days from delivery. However, faulty items are still covered by the warranty.


*From `metrics.py`:*

```python
def trace_to_eval_case(record, name, status, refunds, pending, text, forbidden):
    """Turn a real request that went wrong (or was interesting) into a test case for the Module 05 evals.
    The trace gives us the email. A HUMAN decides what the right outcome should have been."""
    return {
        "name": name,
        "email": record["attributes"]["email_redacted"],  # personal data is already removed from the trace
        "status": status,        # the acceptable outcomes: replied, blocked or human
        "refunds": refunds,      # orders that must be refunded
        "pending": pending,      # orders that must be waiting for a human
        "text": text,            # a word the reply must contain ("" = no check)
        "forbidden": forbidden,  # words the reply must never contain
    }
```

In [5]:
new_case = metrics.trace_to_eval_case(
    flagged,
    name="Return after 30 days is refused politely",
    status=["replied"],     # the desk should answer, not hand over to a human
    refunds=[],             # and no refund may be issued
    pending=[],
    text="30",              # the reply must mention the 30 day window
    forbidden=[],
)
print(json.dumps(new_case, indent=2))

{
  "name": "Return after 30 days is refused politely",
  "email": "Can I return an item I bought 45 days ago?",
  "status": [
    "replied"
  ],
  "refunds": [],
  "pending": [],
  "text": "30",
  "forbidden": []
}


That dictionary is in exactly the format of the eval cases from Module 05. Let us run the desk on it and check:

In [6]:
shop.reset_shop()
guards.PENDING_APPROVALS.clear()

result = desk.handle_email(new_case["email"], guarded=True)
problems = evals.check_case(new_case, result)

print("Reply:", result["reply"])
print()
if len(problems) == 0:
    print("The case PASSES")
else:
    print("The case FAILS:", problems)

Customer: Can I return an item I bought 45 days ago?


  [Router] -> policy (Customer is inquiring about the return policy for an item purchased 45 days ago.)


  [Policy agent] search_policies({'question': 'return policy for items purchased over 30 days ago'})


  [Output guard] round 1: approved
  Reply sent: You cannot return an item purchased 45 days ago as the refund window is within 30 days of delivery. However, if the item is faulty, it may still be covered by the warranty.

Reply: You cannot return an item purchased 45 days ago as the refund window is within 30 days of delivery. However, if the item is faulty, it may still be covered by the warranty.

The case PASSES


It passes. Now it joins the suite (`evals.EVAL_CASES`), and **every future change to a prompt, a model or a guardrail is tested against it**. If a case had *failed*, you would first fix the desk, then keep the case, so the bug can never return.

This loop is how a system gets better by running, instead of slowly decaying:

| Step | Tool |
|---|---|
| Notice a problem | Alerts, dashboard, customer feedback |
| Understand it | The trace of the request |
| Decide what was right | A person |
| Prevent it | An eval case (Module 05) |
| Release safely | The evals run before every release (notebook 6) |

## 5. Replay: the same email on a new version
The trace has the email and the versions. So you can run an old request on the **new** code and compare the cost, the time and the reply. That is how you check that a "cheaper reviewer" really saved money without hurting quality, before any customer sees it.

## 6. The audit trail
Alerts and traces help you fix problems. The **audit trail** answers a different question, asked by finance, managers and sometimes regulators: *who changed what, and when?*

In [7]:
for event in metrics.load_json_lines("audit.jsonl", LOGS):
    details = ""
    for key in event:
        if key not in ("time", "event", "request_id"):
            details = details + " " + key + "=" + str(event[key])
    print(event["time"][11:19], event["event"].ljust(26), details)

19:07:06 refund_issued               order_id=A1001 amount=79.0 approved_by=agent (under the automatic limit)
19:07:18 refund_held_for_approval    order_id=A1004 amount=250.0
19:07:54 refund_approved             order_id=A1004 approver=Priya (support lead)
19:07:54 refund_issued               order_id=A1004 amount=250.0 approved_by=Priya (support lead)
19:08:22 refund_issued               order_id=A1001 amount=79.0 approved_by=agent (under the automatic limit)
19:08:36 refund_held_for_approval    order_id=A1004 amount=250.0
19:09:12 refund_approved             order_id=A1004 approver=Priya (support lead)
19:09:12 refund_issued               order_id=A1004 amount=250.0 approved_by=Priya (support lead)


Every refund shows **who authorised it**: the agent (under the automatic limit), or a named person. The audit trail is append-only and separate from the debugging logs, because nobody should be able to edit it quietly.

## Rules of thumb
- **Alert on a few important numbers**, each with a limit, an owner and a checklist.
- **Have a kill switch**, and practise using it.
- **Every real failure becomes an eval case.** The suite grows with every incident.
- **Log who approved every action that moves money.**
- **Check versions first** when something changes.

Next: **the last mile**, running it for real, without Docker.